In [ ]:
!pip install scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             confusion_matrix, mean_squared_error, r2_score)

In [ ]:
# Models
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.linear_model import Lasso, Ridge

# Metrics
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
df = pd.read_csv('/content/creditcard.csv')
df.head()

In [ ]:
df.info()

In [ ]:
# Assuming last column is target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Take first 1000 features (or all if less)
X = X.iloc[:, :min(1000, X.shape[1])]

print("Features used:", X.shape)

In [ ]:
# 4. DATA CLEANING & TRAIN-TEST SPLIT
# Drop rows with missing values to avoid ValueError
X = X.dropna()
y = y.loc[X.index].dropna()
X = X.loc[y.index]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. SCALING

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Data cleaned, split, and scaled successfully.")

In [ ]:
# =========================
# 6. HANDLE IMBALANCE (IMPORTANT for credit card fraud)
# =========================
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", np.bincount(y_train))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression (L1)": LogisticRegression(penalty='l1', solver='liblinear'),
    "Logistic Regression (L2)": LogisticRegression(penalty='l2'),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "SVM": SVC(kernel='rbf'),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

In [ ]:
# 8. TRAIN & EVALUATE

results = []

for name, model in models.items():
    print("\n==============================")
    print("Model:", name)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Metrics
    report = classification_report(y_test, y_pred, output_dict=True)

    precision = report['weighted avg']['precision']
    recall = report['weighted avg']['recall']
    f1 = report['weighted avg']['f1-score']

    cm = confusion_matrix(y_test, y_pred)

    print("Precision:", precision)
    print("Recall:", recall)
    print("F1-score:", f1)
    print("Confusion Matrix:\n", cm)

    results.append([name, precision, recall, f1])

In [ ]:
import math

n_models = len(models)
cols = 3
rows = math.ceil(n_models / cols)

fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i])
    axes[i].set_title(f'Confusion Matrix: {name}')
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 9. COMPARATIVE ANALYSIS
# =========================
results_df = pd.DataFrame(results, columns=["Model", "Precision", "Recall", "F1-score"])

print("\n==============================")
print("COMPARISON TABLE")
print(results_df.sort_values(by="F1-score", ascending=False))

In [ ]:
# Prepare data for plotting
metrics_df = pd.DataFrame(results, columns=['Model', 'Precision', 'Recall', 'F1-Score'])

# Reshape for seaborn
plot_data = metrics_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=plot_data, x='Model', y='Score', hue='Metric')


for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3, rotation=45)

plt.title('Model Performance Metrics Comparison')
plt.ylim(0.9, 1.05) # Adjusted limit to make room for labels
plt.xticks(rotation=45)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Using the previously calculated average results
avg_df = pd.DataFrame(avg_results)
plot_avg_data = avg_df.melt(id_vars='Model', var_name='Metric Type', value_name='Score')

# Set aesthetic style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 8))

# Create the bar plot
ax = sns.barplot(data=plot_avg_data, x='Model', y='Score', hue='Metric Type', palette='viridis')

# Add straight value labels on top of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=5, fontsize=10, rotation=0)

# Aesthetic improvements
plt.title('Comparison of Model Performance Averages', fontsize=16, pad=20)
plt.xlabel('Machine Learning Models', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.ylim(min(plot_avg_data['Score']) - 0.05, 1.1)  # Space for labels
plt.xticks(rotation=0, fontsize=10) # Set x-axis labels straight
plt.legend(title='Metric Type', bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

plt.tight_layout()
plt.show()

In [ ]:
best_model = results_df.sort_values(by="F1-score", ascending=False).iloc[0]
print("\nBest Model Based on F1-score:")
print(best_model)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


df = pd.read_csv('/content/creditcard.csv')

# Features & Target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Take first 1000 features (or all if less)
X = X.iloc[:, :min(1000, X.shape[1])]

# SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SCALING
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# =========================
# L1 REGULARIZATION MODEL
# =========================
l1_model = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=1.0,
    max_iter=1000
)

l1_model.fit(X_train, y_train)
y_pred_l1 = l1_model.predict(X_test)

# =========================
# L2 REGULARIZATION MODEL
# =========================
l2_model = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    C=1.0,
    max_iter=1000
)

l2_model.fit(X_train, y_train)
y_pred_l2 = l2_model.predict(X_test)


# EVALUATION

print("\n========== L1 REGULARIZATION ==========")
print(classification_report(y_test, y_pred_l1))

print("\n========== L2 REGULARIZATION ==========")
print(classification_report(y_test, y_pred_l2))



# SHOW FEATURE SELECTION EFFECT


l1_zero_weights = np.sum(l1_model.coef_ == 0)
l2_zero_weights = np.sum(l2_model.coef_ == 0)

print("\n===== FEATURE SELECTION COMPARISON =====")
print("Number of zero coefficients (L1):", l1_zero_weights)
print("Number of zero coefficients (L2):", l2_zero_weights)

In [ ]:
# Edit → Clear all outputs

In [ ]:
get_ipython().system('git add .')
get_ipython().system('git commit -m "Clear notebook outputs"')
get_ipython().system('git push')

In [ ]:
import os

for file in os.listdir('/content'):
    print(file)

In [ ]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        if file.endswith('.ipynb'):
            print(os.path.join(root, file))